# Copilot Product Feedback — Ingester (Fabric)

Lands the **Microsoft 365 Copilot user product feedback** export (the thumbs up/down + verbatim
comments users leave on Copilot responses) into the Lakehouse Delta table **`dbo.user_feedback`**,
which the **AI Business Value Dashboard — 1905 Extra** PBIP reads for the **User Feedback** page.

**Why this exists:** the MAC *Product Feedback* report has **no API** — admins can only *view /
export / delete* it from **Microsoft 365 Admin Center → Health → Product Feedback**. So, exactly like
the Power Platform credit-consumption reports, the CSV is **landed into the Lakehouse by a Power
Automate flow** (`3. Fabric/flows/Copilot_ProductFeedback_Email_to_OneLake.json`) and this notebook
turns it into the table the dashboard expects.

```
M365 Admin Center  ─(export)─▶  Power Automate flow  ─▶  Lakehouse Files/product_feedback/*.csv  ─▶  THIS notebook  ─▶  dbo.user_feedback
```

**Column names are preserved verbatim** (`Feedback Id`, `Logs, Attachments`, `AI Context Prompt`, …)
via **Delta column mapping** — the PBIP's `ProductFeedback` query renames them downstream, so they
must match the export exactly. Two columns are **derived**: `Date Submitted Date` (date part of
`Date Submitted UTC`) and `Sentiment` (Positive / Negative from the thumbs/smiley `Feedback Type`).

**Graceful by design** — feedback is *optional*. No file ⇒ an empty, correctly-named table, and the
PBIP's `Enable_ProductFeedback = "Exclude"` toggle keeps the page dormant. **No app registration /
Graph permission required** — this notebook only reads files already in the Lakehouse.

## 1. Configuration

Tag this cell as the pipeline **`parameters`** cell so a Fabric Pipeline can override `WRITE_MODE`.
`TARGET_TABLE` is schema-qualified (`dbo.`) to work with schema-enabled and legacy Lakehouses.

In [ ]:
# === CONFIG ===  (tag this cell as the pipeline `parameters` cell)

# Folder in the attached Lakehouse where the Power Automate flow lands the feedback CSV(s).
SOURCE_DIR   = 'Files/product_feedback'

# Case-insensitive filename glob. Matches the MAC export whether it's named
# 'feedback.csv', 'ProductFeedback_*.csv', 'extracted_user_feedback.csv', etc.
REPORT_GLOB  = '*feedback*'

TARGET_TABLE = 'dbo.user_feedback'

WRITE_MODE   = 'overwrite'   # 'overwrite' = full snapshot; 'append' = keep history (LoadDate added either way)
ADD_LINEAGE  = True          # add SourceFile + LoadDate columns to every row
STRICT       = False         # False = write an empty table when no file is present; True = raise

## 2. Helpers

`_list_matches` resolves the glob against the Lakehouse `Files/` area (local `/lakehouse/default/`
mount). `_read_csv` parses with multiline + quote-escape so verbatim comments survive. `_enrich`
adds the two derived columns. **Note:** unlike the consumption ingester, column names are **not**
sanitised — the spaced names are kept and Delta column mapping carries them to the SQL endpoint.

In [ ]:
import os, re, datetime
from pyspark.sql import functions as F

_LOAD_TS = datetime.datetime.now(datetime.timezone.utc).strftime('%Y-%m-%dT%H:%M:%SZ')

def _local(path):
    return path if path.startswith('/') else f'/lakehouse/default/{path}'

def _list_matches(pattern):
    base = _local(SOURCE_DIR)
    if not os.path.isdir(base):
        return []
    rx = re.compile('^' + re.escape(pattern).replace('\\*', '.*') + '$', re.IGNORECASE)
    hits = [f for f in os.listdir(base) if rx.match(f) and f.lower().endswith('.csv')]
    return [f'{SOURCE_DIR}/{f}' for f in sorted(hits)]

def _read_csv(path):
    return (spark.read
            .option('header', True)
            .option('multiLine', True)
            .option('escape', '"')
            .option('encoding', 'UTF-8')
            .csv(path))

# Sentiment derived from the thumbs / smiley Feedback Type; Date from Date Submitted UTC (MM/dd/yyyy HH:mm:ss).
_POS = ['thumbs up', 'smile', 'like', 'compliment', 'positive']
_NEG = ['thumbs down', 'frown', 'dislike', 'complaint', 'negative']

def _enrich(df):
    cols = [c.lstrip('\ufeff') for c in df.columns]
    df = df.toDF(*cols)                      # strip any BOM, keep spaces/case
    if 'Feedback Type' in df.columns:
        ft = F.lower(F.trim(F.col('`Feedback Type`')))
        df = df.withColumn('Sentiment',
                           F.when(ft.isin(_POS), F.lit('Positive'))
                            .when(ft.isin(_NEG), F.lit('Negative'))
                            .otherwise(F.lit(None).cast('string')))
    if 'Date Submitted UTC' in df.columns:
        c = F.col('`Date Submitted UTC`')
        ts = F.coalesce(
            F.to_timestamp(c, 'MM/dd/yyyy HH:mm:ss'),
            F.to_timestamp(c, 'M/d/yyyy H:mm:ss'),
            F.to_timestamp(c, 'yyyy-MM-dd HH:mm:ss'),
            F.to_timestamp(c))
        df = df.withColumn('Date Submitted Date', ts.cast('date'))
    return df

print(f'Load timestamp: {_LOAD_TS}')
print(f'Source folder : {SOURCE_DIR}  (exists: {os.path.isdir(_local(SOURCE_DIR))})')

## 3. Ingest → `dbo.user_feedback`

Unions every matching CSV, enriches, adds lineage, and writes with **Delta column mapping** so the
spaced/comma column names (`Feedback Id`, `Logs, Attachments`, …) survive to the SQL endpoint. A
missing file writes an **empty, correctly-named** table so the PBIP query never errors.

In [ ]:
from pyspark.sql.types import StructType, StructField, StringType

# The 23-column contract the PBIP 'ProductFeedback' query expects (names kept verbatim).
EMPTY_SCHEMA = ['Feedback Id', 'Comment', 'Translated Comment', 'Comment Language', 'Date Submitted UTC',
    'Feedback Type', 'Microsoft Response Status', 'App', 'App Language', 'Platform', 'Source Type',
    'Logs, Attachments', 'User Id', 'User Email', 'Browser', 'Browser Version', 'AI Context Prompt',
    'AI Context Response Message', 'Survey Question', 'Survey Response Option', 'Additional Metadata',
    'Date Submitted Date', 'Sentiment']

def _empty(cols):
    cols = list(cols) + (['SourceFile', 'LoadDate'] if ADD_LINEAGE else [])
    return spark.createDataFrame([], StructType([StructField(c, StringType(), True) for c in cols]))

def _write(df, table):
    (df.write.mode(WRITE_MODE)
        .option('overwriteSchema', 'true')
        .option('delta.columnMapping.mode', 'name')      # allow spaces/commas in column names
        .option('delta.minReaderVersion', '2')
        .option('delta.minWriterVersion', '5')
        .format('delta').saveAsTable(table))

matches = _list_matches(REPORT_GLOB)
if not matches:
    msg = f'no file matched "{REPORT_GLOB}" in {SOURCE_DIR}'
    if STRICT:
        raise FileNotFoundError(msg)
    print(f'\u26a0  {msg}; writing EMPTY {TARGET_TABLE} so the PBIP still resolves.')
    _write(_empty(EMPTY_SCHEMA), TARGET_TABLE)
    rows = 0
else:
    frames = []
    for path in matches:
        d = _enrich(_read_csv(path))
        if ADD_LINEAGE:
            d = d.withColumn('SourceFile', F.lit(os.path.basename(path))) \
                 .withColumn('LoadDate',   F.lit(_LOAD_TS))
        frames.append(d)
    df = frames[0]
    for d in frames[1:]:
        df = df.unionByName(d, allowMissingColumns=True)
    # De-duplicate on Feedback Id (the PK). Overlapping export files (e.g. monthly
    # pulls) repeat rows, and the model uses Feedback Id as the one-side of a
    # relationship - duplicates make the Power BI load fail.
    if 'Feedback Id' in df.columns:
        df = df.dropDuplicates(['Feedback Id'])
    # Guarantee the full feedback column contract regardless of which OCV export variant
    # the customer has, or of Spark's multiline-CSV parser dropping/shifting columns on
    # malformed rows (embedded HTML/newlines): any contract column the parse didn't
    # produce is added as null. The PBIP's ProductFeedback query then always resolves
    # every column it renames/selects (e.g. 'User Id'), so the dashboard never errors
    # with \"the column ... wasn't found\" on an inconsistent export.
    for _c in EMPTY_SCHEMA:
        if _c not in df.columns:
            df = df.withColumn(_c, F.lit(None).cast('string'))
    rows = df.count()
    _write(df, TARGET_TABLE)
    print(f'\u2713  {TARGET_TABLE}: {len(matches)} file(s), {rows:,} rows -> written ({WRITE_MODE})')

print('Done. Refresh the PBIP / Direct Lake model to pick up user_feedback.')

## 4. Verify

Headline counts + the thumbs/sentiment split, so you can sanity-check the load.

In [ ]:
try:
    fb = spark.table(TARGET_TABLE)
    n = fb.count()
    print(f'{TARGET_TABLE}: {n:,} rows, {len(fb.columns)} columns')
    if n:
        fb.groupBy('`Feedback Type`', 'Sentiment').count().orderBy(F.desc('count')).show(truncate=False)
        fb.select(F.min('`Date Submitted Date`').alias('first'),
                  F.max('`Date Submitted Date`').alias('last')).show()
except Exception as e:
    print('verify skipped:', e)